# Test04: LangChain doctor and judge

This research experiment uses local ICD search, TF-IDF cosine similarity, full four-character ICD families, and similar **labeled training notes**. The 10 test notes are held out from retrieval. It is not for clinical use.

Install the dependencies into the **selected notebook kernel** with `python -m pip install -r requirements_test04.txt`. The default below is an offline retrieval baseline. Set `run_mode = 'agents'` to run the LangChain agents: this sends test notes, retrieved training examples, and ICD context to OpenAI and incurs API charges. Confirm that you are permitted to share the notes first. The model-call cap is not a dollar budget, and this experiment is not guaranteed to meet the assessment's $1/60-minute limits. If your network uses a TLS-inspecting proxy, configure its trusted CA for the selected Python environment; do not disable certificate verification.

In [4]:
from pathlib import Path
import html
import json
import subprocess
import sys
from IPython.display import HTML, display

root = Path.cwd()
if not (root / 'test04_agents.py').is_file():
    raise FileNotFoundError('Start this notebook with the project root as the working directory.')
out = root / 'out'
out.mkdir(exist_ok=True)
run_mode = 'agents'  # Change to 'agents' to use OpenAI.
doctor_model = None  # None uses OPENAI_MODEL or gpt-6-luna.
judge_model = None   # None uses the doctor model.
max_retries = 3      # Initial doctor attempt plus up to three retries.
max_model_calls = 5  # Maximum calls per agent invocation, not a dollar cap.
if run_mode not in {'offline', 'agents'}:
    raise ValueError("run_mode must be 'offline' or 'agents'")
label = 'offline_' if run_mode == 'offline' else ''
predictions_path = out / f'dev_predictions_{label}test04.jsonl'
reviews_path = out / f'dev_reviews_{label}test04.jsonl'
per_note_path = out / f'dev_per_note_{label}test04.jsonl'

## Generate predictions

Each completed note is written immediately. Rerunning this cell overwrites the selected mode's Test04 output files. In agent mode, if no `OPENAI_API_KEY` exists in the environment or project `.env`, the workflow explicitly falls back to local retrieval. Agent failures stop the run and preserve completed rows.

In [5]:
command = [sys.executable, str(root / 'test04_agents.py'),
           '--predictions', str(predictions_path), '--reviews', str(reviews_path),
           '--max-retries', str(max_retries), '--max-model-calls', str(max_model_calls)]
if run_mode == 'offline':
    command.append('--offline')
if doctor_model:
    command.extend(['--doctor-model', doctor_model])
if judge_model:
    command.extend(['--judge-model', judge_model])
_ = subprocess.run(command, cwd=root, check=True)

Agent mode: doctor=gpt-6-luna, judge=gpt-6-luna; notes and tool results are sent to OpenAI.
[1/10] dev-051
  Attempt 1: doctor agent
  Doctor: 3 model calls; tools: ['search_icd', 'training_examples', 'icd_context', 'search_icd', 'compare_similarity']
  Attempt 1: judge agent
  Judge: 2 model calls; tools: ['search_icd']
[2/10] dev-052
  Attempt 1: doctor agent
  Doctor: 3 model calls; tools: ['search_icd', 'training_examples', 'icd_context', 'icd_context', 'icd_context', 'compare_similarity']
  Attempt 1: judge agent
  Judge: 2 model calls; tools: ['icd_context']
[3/10] dev-053
  Attempt 1: doctor agent
  Doctor: 3 model calls; tools: ['search_icd', 'training_examples', 'icd_context', 'compare_similarity']
  Attempt 1: judge agent
  Judge: 2 model calls; tools: ['icd_context']
[4/10] dev-054
  Attempt 1: doctor agent
  Doctor: 3 model calls; tools: ['search_icd', 'training_examples', 'icd_context', 'icd_context', 'compare_similarity']
  Attempt 1: judge agent
  Judge: 2 model calls; t

## Score held-out notes

The scorer accepts up to three valid ranked MMS codes. The 10 cases have been used in prior experiments, so this is an exploratory evaluation, not an untouched final test.

In [6]:
_ = subprocess.run([sys.executable, str(root / 'score.py'),
                '--gold', str(root / 'data/dev_notes_test.jsonl'),
                '--pred', str(predictions_path),
                '--mms', str(root / 'data/mms-2026-01.json'),
                '--per-note', str(per_note_path)], cwd=root, check=True)

notes 10  missing predictions 0  invalid codes 0  duplicate ids 0
slice A: n=10  top1 0.800  top3 0.900  points 62.5 / 75
SCORE 62.5 / 75


## Review results

`approved` is `None` for offline results. `complete` means three codes were returned, but an approved list can legitimately contain fewer. Review files include feedback and tool names, not model transcripts.

In [7]:
with per_note_path.open(encoding='utf-8') as file:
    results = [json.loads(line) for line in file if line.strip()]
with reviews_path.open(encoding='utf-8') as file:
    reviews = {row['id']: row for row in (json.loads(line) for line in file if line.strip())}
columns = ('id', 'gold', 'pred', 'top1', 'top3', 'mode', 'approved', 'complete', 'attempts')
header = ''.join(f'<th>{html.escape(column)}</th>' for column in columns)
rows = []
for result in results:
    row = {**result, **reviews.get(result['id'], {})}
    cells = ''.join(f'<td>{html.escape(str(row.get(column, "")))}</td>' for column in columns)
    rows.append(f'<tr>{cells}</tr>')
print(f'{len(results)} notes evaluated; modes: {sorted({row["mode"] for row in reviews.values()})}')
display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{"".join(rows)}</tbody></table>'))

10 notes evaluated; modes: ['agents']


id,gold,pred,top1,top3,mode,approved,complete,attempts
dev-051,['5B52'],"['5B52', 'MG29.3']",True,True,agents,True,False,"[{'attempt': 1, 'codes': ['5B52', 'MG29.3'], 'approved': True, 'status': 'approved', 'feedback': ""Approve the ranked list. Primary 5B52 is supported by the clinician's explicit diagnosis of severe acute malnutrition in a child, with MUAC 11.0 cm and bilateral nutritional oedema documented. Though the supplied definition mentions weight-for-height Z score, the note does not report that measurement; the clinician's explicit SAM diagnosis and MUAC/oedema findings support the broad acute-malnutrition category. MG29.3 is a supported alternative for the explicitly documented bilateral pitting oedema. No more specific oedema subtype is warranted because the swelling is described at the feet and there is no basis to infer ankle oedema or another cause."", 'doctor_tools': ['search_icd', 'training_examples', 'icd_context', 'search_icd', 'compare_similarity'], 'doctor_model_calls': 3, 'judge_tools': ['search_icd'], 'judge_model_calls': 2}]"
dev-052,['BD10'],"['BD10', 'BD1Z']",True,True,agents,True,False,"[{'attempt': 1, 'codes': ['BD10', 'BD1Z'], 'approved': True, 'status': 'approved', 'feedback': 'Approve BD10 (Congestive heart failure) as the supported primary code. The note explicitly diagnoses congestive heart failure and documents active fluid congestion (orthopnea, paroxysmal nocturnal dyspnea, raised neck veins, crackles, peripheral edema, hepatomegaly, and weight gain). BD1Z is a valid alternative but less specific and is not needed when the congestive type is documented. No ejection-fraction or echocardiographic detail is available, so do not infer a more specific subtype.', 'doctor_tools': ['search_icd', 'training_examples', 'icd_context', 'icd_context', 'icd_context', 'compare_similarity'], 'doctor_model_calls': 3, 'judge_tools': ['icd_context'], 'judge_model_calls': 2}]"
dev-053,['QA42.0'],['QA42.0'],True,True,agents,True,False,"[{'attempt': 1, 'codes': ['QA42.0'], 'approved': True, 'status': 'approved', 'feedback': 'QA42.0 — Supervision of normal first pregnancy — is supported: this is an ANC supervision visit, explicitly documented as a normal, low-risk first pregnancy (G1P0), with no complications or abnormalities reported. Exact decimal specificity is appropriate. No alternatives are needed.', 'doctor_tools': ['search_icd', 'training_examples', 'icd_context', 'compare_similarity'], 'doctor_model_calls': 3, 'judge_tools': ['icd_context'], 'judge_model_calls': 2}]"
dev-054,['5A40.0'],"['5A40.0', 'MA18.0Y']",True,True,agents,True,False,"[{'attempt': 1, 'codes': ['5A40.0', 'MA18.0Y'], 'approved': True, 'status': 'approved', 'feedback': 'Approve the ranked list. 5A40.0 (Impaired fasting glucose) is the best-supported primary code: the clinician documents prediabetes/raised fasting sugar and repeated fasting glucose values of 6.4 and 6.6 mmol/L, within the code definition’s 6.1–6.9 mmol/L range and below the diabetes threshold. MA18.0Y is an eligible but less specific alternative; the documented fasting abnormality is specifically classified by 5A40.0, so retain the proposed ranking.', 'doctor_tools': ['search_icd', 'training_examples', 'icd_context', 'icd_context', 'compare_similarity'], 'doctor_model_calls': 3, 'judge_tools': ['search_icd'], 'judge_model_calls': 2}]"
dev-055,['BA40.1'],['BA40.1'],True,True,agents,True,False,"[{'attempt': 1, 'codes': ['BA40.1'], 'approved': True, 'status': 'approved', 'feedback': 'BA40.1 (Stable angina) is supported: the note documents recurrent exertional chest tightness with a stable pattern, promptly relieved by rest, without rest or crescendo symptoms. It is MMS-valid and at the appropriate specificity. No alternatives are needed.', 'doctor_tools': ['search_icd', 'training_examples', 'icd_context', 'icd_context', 'icd_context', 'search_icd', 'compare_similarity'], 'doctor_model_calls': 4, 'judge_tools': ['icd_context'], 'judge_model_calls': 2}]"
dev-056,['JA63.2'],['JA